# Vectors and Spaces - Exercises

**Chapter 02, Section 01** | [notes.md](notes.md) | [theory.ipynb](theory.ipynb)

This notebook is designed as the practice counterpart to the chapter and theory notebook.

The exercise path is ordered for learning:
1. vector operations and geometry
2. span, basis, independence, and coordinates
3. projection, orthogonality, and least squares
4. simplex geometry and attention
5. high-dimensional behavior
6. linear maps, low rank, and AI-specific structure
7. regularization geometry

This version is intentionally harder than a step-by-step coding worksheet. Many exercises now include a short written or proof-style component before the implementation.

Each exercise has:
1. a problem statement
2. a scaffold cell with TODOs
3. a complete solution cell

Primary ideas used here come directly from the notes and theory notebook, with the same conceptual backbone: vectors, subspaces, projections, high-dimensional geometry, attention, LoRA, and NTK-style inner-product thinking.

In [28]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)


def svd_null_space(A, tol=1e-10):
    A = np.asarray(A, dtype=float)
    _, S, Vt = np.linalg.svd(A, full_matrices=True)
    rank = np.sum(S > tol)
    return Vt[rank:].T


def pairwise_distances(X):
    X = np.asarray(X, dtype=float)
    G = X @ X.T
    diag = np.diag(G)
    sq = np.maximum(diag[:, None] + diag[None, :] - 2 * G, 0.0)
    return np.sqrt(sq)


---
## Exercise 1: Dot Product and p-Norms

**Task**: Implement two of the most basic primitives in linear algebra.

1. `dot_product(u, v)` using elementwise multiplication and a sum
2. `p_norm(v, p)` for `p = 1, 2, 3, ...`

**Requirements**:
- Do not use `np.dot` inside `dot_product`
- Do not use `np.linalg.norm` inside `p_norm`
- Verify Cauchy-Schwarz numerically on one example

**Written part**:
- Explain why `p < 1` does not define a norm in general.

In [2]:
# === Exercise 1: Scaffold ===

def dot_product(u, v):
    """Return sum_i u_i v_i."""
    pass  # YOUR CODE HERE


def p_norm(v, p=2):
    """Return (sum_i |v_i|^p)^(1/p). Assume p >= 1."""
    pass  # YOUR CODE HERE


# Build your own nontrivial test cases.
# Include at least one sign-changing vector and one Cauchy-Schwarz check.

In [29]:
# === Exercise 1: Solution ===

def dot_product(u, v):
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)
    return float(np.sum(u * v))


def p_norm(v, p=2):
    v = np.asarray(v, dtype=float)
    return float(np.sum(np.abs(v) ** p) ** (1.0 / p))


u = np.array([1.0, 2.0, -1.0])
v = np.array([3.0, 0.0, 1.0])

print("u.v =", dot_product(u, v))
for p in [1, 2, 3]:
    print(f"||u||_{p} = {p_norm(u, p):.6f}")

lhs = abs(dot_product(u, v))
rhs = p_norm(u, 2) * p_norm(v, 2)
print("Cauchy-Schwarz check:", lhs, "<=", rhs, "->", lhs <= rhs + 1e-12)

u.v = 2.0
||u||_1 = 4.000000
||u||_2 = 2.449490
||u||_3 = 2.154435
Cauchy-Schwarz check: 2.0 <= 7.745966692414833 -> True


---
## Exercise 2: Cosine Similarity and Angle

**Task**: Implement directional similarity and convert it into an angle.

**Requirements**:
- Implement `cosine_similarity(u, v)`
- Implement `angle_between(u, v, degrees=True)`
- Verify for unit vectors that `||u-v||^2 = 2 - 2 cos(u,v)`

**Written part**:
- Explain why cosine similarity can stay high even when Euclidean distance is large.

In [4]:
# === Exercise 2: Scaffold ===

def cosine_similarity(u, v):
    pass  # YOUR CODE HERE


def angle_between(u, v, degrees=True):
    pass  # YOUR CODE HERE


# Choose at least one orthogonal pair and one non-orthogonal pair.
# Use your own tests to verify the unit-vector identity.

In [5]:
# === Exercise 2: Solution ===

def cosine_similarity(u, v):
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)
    return float(np.sum(u * v) / (np.sqrt(np.sum(u ** 2)) * np.sqrt(np.sum(v ** 2))))


def angle_between(u, v, degrees=True):
    cos = np.clip(cosine_similarity(u, v), -1.0, 1.0)
    angle = np.arccos(cos)
    return float(np.degrees(angle) if degrees else angle)


u = np.array([1.0, 1.0])
v = np.array([1.0, -1.0])
u = u / np.linalg.norm(u)
v = v / np.linalg.norm(v)

cos = cosine_similarity(u, v)
dist_sq = np.sum((u - v) ** 2)
rhs = 2 - 2 * cos

print("cos(u,v) =", cos)
print("angle(u,v) =", angle_between(u, v), "degrees")
print("||u-v||^2 =", dist_sq)
print("2 - 2 cos(u,v) =", rhs)
print("identity holds:", np.allclose(dist_sq, rhs))

cos(u,v) = 0.0
angle(u,v) = 90.0 degrees
||u-v||^2 = 1.9999999999999996
2 - 2 cos(u,v) = 2.0
identity holds: True


---
## Exercise 3: Span, Linear Independence, and Coordinates

**Task**: Work with span and basis coordinates directly.

**Requirements**:
- Implement `are_linearly_independent(vectors)` using rank
- Implement `coordinates_in_basis(B, x)` for a square basis matrix `B`
- Implement `is_in_span(vectors, target)` using rank or least squares

**Written part**:
- Explain why coordinate vectors are unique only when the basis vectors are linearly independent.

In [6]:
# === Exercise 3: Scaffold ===

def are_linearly_independent(vectors):
    pass  # YOUR CODE HERE


def coordinates_in_basis(B, x):
    pass  # YOUR CODE HERE


def is_in_span(vectors, target, tol=1e-10):
    pass  # YOUR CODE HERE


# Provide one independent example, one dependent example, and one coordinate-recovery example.

In [7]:
# === Exercise 3: Solution ===

def are_linearly_independent(vectors):
    A = np.column_stack(vectors).astype(float)
    return np.linalg.matrix_rank(A) == A.shape[1]


def coordinates_in_basis(B, x):
    B = np.asarray(B, dtype=float)
    x = np.asarray(x, dtype=float)
    return np.linalg.solve(B, x)


def is_in_span(vectors, target, tol=1e-10):
    A = np.column_stack(vectors).astype(float)
    target = np.asarray(target, dtype=float)
    coeffs, *_ = np.linalg.lstsq(A, target, rcond=None)
    residual = np.linalg.norm(A @ coeffs - target)
    return residual < tol, coeffs


independent = [np.array([1.0, 0.0]), np.array([0.0, 1.0])]
dependent = [np.array([1.0, 2.0]), np.array([2.0, 4.0])]
B = np.array([[1.0, 1.0], [1.0, -1.0]])
x = np.array([4.0, 2.0])

print("independent set?", are_linearly_independent(independent))
print("dependent set?", are_linearly_independent(dependent))
print("coordinates of x in basis B =", coordinates_in_basis(B, x))
in_span, coeffs = is_in_span(independent, np.array([7.0, 4.0]))
print("[7,4] in span(e1,e2)?", in_span, "with coeffs", coeffs)

independent set? True
dependent set? False
coordinates of x in basis B = [3. 1.]
[7,4] in span(e1,e2)? True with coeffs [7. 4.]


---
## Exercise 4: Projection and Gram-Schmidt

**Task**: Move from algebra to geometry.

**Requirements**:
- Implement `project_onto(u, v)` to project `u` onto the line spanned by `v`
- Implement `gram_schmidt(vectors)` returning orthonormal vectors
- Verify orthonormality numerically

**Written part**:
- Prove or explain why a set of nonzero pairwise-orthogonal vectors must be linearly independent.

In [8]:
# === Exercise 4: Scaffold ===

def project_onto(u, v):
    pass  # YOUR CODE HERE


def gram_schmidt(vectors, tol=1e-10):
    pass  # YOUR CODE HERE


# Choose a projection example where the residual is visibly nonzero.
# Then choose a spanning set with redundant overlap and orthonormalize it.

In [9]:
# === Exercise 4: Solution ===

def project_onto(u, v):
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)
    return (np.dot(u, v) / np.dot(v, v)) * v


def gram_schmidt(vectors, tol=1e-10):
    orthonormal = []
    for v in vectors:
        w = np.array(v, dtype=float)
        for u in orthonormal:
            w = w - np.dot(w, u) * u
        n = np.linalg.norm(w)
        if n > tol:
            orthonormal.append(w / n)
    return np.array(orthonormal)


u = np.array([1.0, 2.0, -1.0])
v = np.array([3.0, 0.0, 1.0])
proj = project_onto(u, v)
resid = u - proj

Q = gram_schmidt([
    np.array([1.0, 1.0, 0.0]),
    np.array([1.0, 0.0, 1.0]),
    np.array([0.0, 1.0, 1.0])
])

print("projection =", proj)
print("residual orthogonal to v:", np.allclose(np.dot(resid, v), 0.0))
print("\nGram-Schmidt output =\n", Q)
print("Q Q^T =\n", Q @ Q.T)

projection = [0.6 0.  0.2]
residual orthogonal to v: True

Gram-Schmidt output =
 [[ 0.7071  0.7071  0.    ]
 [ 0.4082 -0.4082  0.8165]
 [-0.5774  0.5774  0.5774]]
Q Q^T =
 [[ 1.  0.  0.]
 [ 0.  1. -0.]
 [ 0. -0.  1.]]


---
## Exercise 5: Projection Matrix and Least Squares

**Task**: Implement projection onto a column space and solve a least-squares problem.

**Requirements**:
- Implement `projection_matrix(A) = A (A^T A)^(-1) A^T`
- Implement `least_squares_solution(A, b)`
- Verify the residual is orthogonal to the column space

**Written part**:
- Explain why the least-squares residual must lie in the orthogonal complement of `col(A)`.

In [10]:
# === Exercise 5: Scaffold ===

def projection_matrix(A):
    pass  # YOUR CODE HERE


def least_squares_solution(A, b):
    pass  # YOUR CODE HERE


# Pick a full-column-rank matrix A and a target vector b.
# Your tests should verify idempotence and the normal equations.

In [11]:
# === Exercise 5: Solution ===

def projection_matrix(A):
    A = np.asarray(A, dtype=float)
    return A @ np.linalg.inv(A.T @ A) @ A.T


def least_squares_solution(A, b):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    return np.linalg.solve(A.T @ A, A.T @ b)


A = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
b = np.array([2.0, 1.0, 4.0])
P = projection_matrix(A)
x_star = least_squares_solution(A, b)
residual = A @ x_star - b

print("P =\n", P)
print("P^2 = P?", np.allclose(P @ P, P))
print("x* =", x_star)
print("Ax* =", A @ x_star)
print("A^T(Ax*-b) =", A.T @ residual)

P =
 [[ 0.6667 -0.3333  0.3333]
 [-0.3333  0.6667  0.3333]
 [ 0.3333  0.3333  0.6667]]
P^2 = P? True
x* = [2.3333 1.3333]
Ax* = [2.3333 1.3333 3.6667]
A^T(Ax*-b) = [0. 0.]


---
## Exercise 6: Rank-Nullity and the Four Fundamental Subspaces

**Task**: Use SVD to recover rank, nullity, right null space, and left null space.

**Requirements**:
- Implement `rank_nullity(A)`
- Implement `null_space_basis(A)`
- Implement `left_null_space_basis(A)`
- Verify `rank + nullity = n`

**Written part**:
- Describe geometrically what the null space and image of a linear map mean.

In [12]:
# === Exercise 6: Scaffold ===

def rank_nullity(A, tol=1e-10):
    pass  # YOUR CODE HERE


def null_space_basis(A, tol=1e-10):
    pass  # YOUR CODE HERE


def left_null_space_basis(A, tol=1e-10):
    pass  # YOUR CODE HERE


# Use a rank-deficient matrix with a nontrivial right null space.
# Include checks for both right and left null-space relations.

In [13]:
# === Exercise 6: Solution ===

def rank_nullity(A, tol=1e-10):
    A = np.asarray(A, dtype=float)
    S = np.linalg.svd(A, compute_uv=False)
    rank = int(np.sum(S > tol))
    nullity = A.shape[1] - rank
    return rank, nullity


def null_space_basis(A, tol=1e-10):
    return svd_null_space(A, tol=tol)


def left_null_space_basis(A, tol=1e-10):
    return svd_null_space(np.asarray(A, dtype=float).T, tol=tol)


A = np.array([
    [1.0, 2.0, 3.0],
    [2.0, 4.0, 6.0],
    [1.0, 1.0, 2.0]
])
rank, nullity = rank_nullity(A)
null_basis = null_space_basis(A)
left_null_basis = left_null_space_basis(A)

print("rank =", rank, "nullity =", nullity)
print("rank + nullity =", rank + nullity, "domain dimension =", A.shape[1])
print("null basis =\n", null_basis)
print("left null basis =\n", left_null_basis)
if null_basis.size:
    print("A @ null_basis =\n", A @ null_basis)
if left_null_basis.size:
    print("left_null_basis^T @ A =\n", left_null_basis.T @ A)

rank = 2 nullity = 1
rank + nullity = 3 domain dimension = 3
null basis =
 [[ 0.5774]
 [ 0.5774]
 [-0.5774]]
left null basis =
 [[ 0.8944]
 [-0.4472]
 [ 0.    ]]
A @ null_basis =
 [[0.]
 [0.]
 [0.]]
left_null_basis^T @ A =
 [[0. 0. 0.]]


---
## Exercise 7: Softmax, the Simplex, and Attention Output

**Task**: Show explicitly that attention weights form a probability vector and that the output is a convex combination of value vectors.

**Requirements**:
- Implement a numerically stable `softmax(logits, tau)`
- Implement `is_in_simplex(p)`
- Implement `attention_output(logits, values, tau)`
- Compare low temperature and high temperature behavior

**Written part**:
- Explain why single-head attention output must lie in the convex hull of the value vectors.

In [14]:
# === Exercise 7: Scaffold ===

def softmax(logits, tau=1.0):
    pass  # YOUR CODE HERE


def is_in_simplex(p, tol=1e-10):
    pass  # YOUR CODE HERE


def attention_output(logits, values, tau=1.0):
    pass  # YOUR CODE HERE


# Choose a small collection of value vectors and two very different temperatures.
# Your test should make the geometric effect of temperature obvious.

In [15]:
# === Exercise 7: Solution ===

def softmax(logits, tau=1.0):
    z = np.asarray(logits, dtype=float) / tau
    z = z - np.max(z)
    e = np.exp(z)
    return e / np.sum(e)


def is_in_simplex(p, tol=1e-10):
    p = np.asarray(p, dtype=float)
    return bool(np.all(p >= -tol) and abs(np.sum(p) - 1.0) < tol)


def attention_output(logits, values, tau=1.0):
    weights = softmax(logits, tau=tau)
    values = np.asarray(values, dtype=float)
    return weights, weights @ values


logits = np.array([2.5, 1.0, -0.5])
values = np.array([[2.0, 0.0], [0.0, 2.0], [1.0, 1.0]])

for tau in [0.2, 1.0, 5.0]:
    weights, out = attention_output(logits, values, tau=tau)
    print(f"tau={tau:>3}: weights={weights}, simplex={is_in_simplex(weights)}, output={out}")

tau=0.2: weights=[0.9994 0.0006 0.    ], simplex=True, output=[1.9989 0.0011]
tau=1.0: weights=[0.7856 0.1753 0.0391], simplex=True, output=[1.6103 0.3897]
tau=5.0: weights=[0.4368 0.3236 0.2397], simplex=True, output=[1.1132 0.8868]


---
## Exercise 8: High-Dimensional Random Geometry

**Task**: Empirically verify that random unit vectors become nearly orthogonal as dimension grows.

**Requirements**:
- Implement `random_unit_vectors(n, d)`
- Implement `pairwise_cosine_stats(X)` returning mean and std of off-diagonal cosines
- Compare dimensions `d = 4, 32, 256`

**Written part**:
- Predict qualitatively how the standard deviation of pairwise cosine similarity should scale with dimension.

In [16]:
# === Exercise 8: Scaffold ===

def random_unit_vectors(n, d):
    pass  # YOUR CODE HERE


def pairwise_cosine_stats(X):
    pass  # YOUR CODE HERE


# Run the same experiment across several dimensions and compare the concentration trend.

In [17]:
# === Exercise 8: Solution ===

def random_unit_vectors(n, d):
    X = np.random.randn(n, d)
    X /= np.linalg.norm(X, axis=1, keepdims=True)
    return X


def pairwise_cosine_stats(X):
    G = X @ X.T
    upper = G[np.triu_indices_from(G, k=1)]
    return float(np.mean(upper)), float(np.std(upper))


for d in [4, 32, 256]:
    X = random_unit_vectors(300, d)
    mean_cos, std_cos = pairwise_cosine_stats(X)
    print(f"d={d:>3} -> mean cosine = {mean_cos:+.4f}, std = {std_cos:.4f}")

d=  4 -> mean cosine = -0.0018, std = 0.4990
d= 32 -> mean cosine = -0.0007, std = 0.1768
d=256 -> mean cosine = -0.0001, std = 0.0625


---
## Exercise 9: Johnson-Lindenstrauss Style Random Projection

**Task**: Implement a random Gaussian projection and check how well pairwise distances are preserved.

**Requirements**:
- Implement `jl_project(X, k, seed=0)`
- Project a high-dimensional point cloud into a much lower-dimensional space
- Compute median relative distortion of pairwise distances

**Written part**:
- Explain why the Johnson-Lindenstrauss target dimension depends on `log n` rather than the original ambient dimension.

In [18]:
# === Exercise 9: Scaffold ===

def jl_project(X, k, seed=0):
    pass  # YOUR CODE HERE


# Use a random Gaussian projection and summarize the distance distortion statistics.

In [19]:
# === Exercise 9: Solution ===

def jl_project(X, k, seed=0):
    rng = np.random.default_rng(seed)
    X = np.asarray(X, dtype=float)
    R = rng.standard_normal((X.shape[1], k)) / np.sqrt(k)
    return X @ R


X = np.random.randn(80, 300)
Y = jl_project(X, k=40, seed=7)

D_X = pairwise_distances(X[:30])
D_Y = pairwise_distances(Y[:30])
mask = np.triu(np.ones_like(D_X, dtype=bool), k=1)
distortion = np.abs(D_Y[mask] - D_X[mask]) / np.maximum(D_X[mask], 1e-12)

print("median relative distortion =", np.median(distortion))
print("90th percentile distortion =", np.percentile(distortion, 90))
print("max distortion =", np.max(distortion))

median relative distortion = 0.06824817795657001
90th percentile distortion = 0.16476167465678404
max distortion = 0.3515867906883106


---
## Exercise 10: Attention Bilinear Form and LoRA Low Rank

**Task**: Connect linear algebra directly to Transformer internals.

**Requirements**:
- Implement `attention_interaction_rank(W_Q, W_K)` returning rank of `W_Q W_K^T`
- Implement `lora_update(B, A)` returning `B @ A`
- Verify the ranks are bounded by the head dimension / LoRA rank

**Written part**:
- Explain why both attention bilinear forms and LoRA updates are naturally low-rank objects.

In [20]:
# === Exercise 10: Scaffold ===

def attention_interaction_rank(W_Q, W_K):
    pass  # YOUR CODE HERE


def lora_update(B, A):
    pass  # YOUR CODE HERE


# Choose dimensions where the rank upper bound is visibly nontrivial and verify it.

In [21]:
# === Exercise 10: Solution ===

def attention_interaction_rank(W_Q, W_K):
    return int(np.linalg.matrix_rank(W_Q @ W_K.T))


def lora_update(B, A):
    return B @ A


d, d_k, r = 64, 8, 4
W_Q = np.random.randn(d, d_k)
W_K = np.random.randn(d, d_k)
interaction_rank = attention_interaction_rank(W_Q, W_K)

B = np.random.randn(d, r)
A = np.random.randn(r, d)
delta = lora_update(B, A)
delta_rank = np.linalg.matrix_rank(delta)

print("rank(W_Q W_K^T) =", interaction_rank, "<=", d_k)
print("rank(BA) =", delta_rank, "<=", r)

rank(W_Q W_K^T) = 8 <= 8
rank(BA) = 4 <= 4


---
## Exercise 11: Soft Thresholding and L2 Shrinkage

**Task**: Compare the geometric effect of L1 and L2 regularization on coordinates.

**Requirements**:
- Implement `soft_threshold(x, lam)` (the L1 proximal operator)
- Implement `l2_shrink(x, lam)` using the simple shrink rule `x / (1 + lam)`
- Show numerically that L1 can create exact zeros while L2 generally does not

**Written part**:
- Explain the corner-vs-sphere geometry behind sparsity for L1 regularization.

In [22]:
# === Exercise 11: Scaffold ===

def soft_threshold(x, lam):
    pass  # YOUR CODE HERE


def l2_shrink(x, lam):
    pass  # YOUR CODE HERE


# Compare the two operators on a vector containing both large and near-zero coordinates.

In [23]:
# === Exercise 11: Solution ===

def soft_threshold(x, lam):
    x = np.asarray(x, dtype=float)
    return np.sign(x) * np.maximum(np.abs(x) - lam, 0.0)


def l2_shrink(x, lam):
    x = np.asarray(x, dtype=float)
    return x / (1.0 + lam)


x = np.array([2.0, -0.3, 0.08, -4.0])
lam = 0.25
x_l1 = soft_threshold(x, lam)
x_l2 = l2_shrink(x, lam)

print("original =", x)
print("L1 soft-thresholded =", x_l1)
print("L2 shrunk =", x_l2)
print("number of exact zeros under L1 =", np.sum(np.isclose(x_l1, 0.0)))
print("number of exact zeros under L2 =", np.sum(np.isclose(x_l2, 0.0)))

original = [ 2.   -0.3   0.08 -4.  ]
L1 soft-thresholded = [ 1.75 -0.05  0.   -3.75]
L2 shrunk = [ 1.6   -0.24   0.064 -3.2  ]
number of exact zeros under L1 = 1
number of exact zeros under L2 = 0


---
## Exercise 12: Parameter Gradients and a Tiny NTK

**Task**: Show the NTK idea in the simplest possible model.

For a linear model `f_theta(x) = theta^T x`, the gradient with respect to parameters is exactly `x`, so the kernel is just the Gram matrix `X X^T`.

**Requirements**:
- Implement `linear_ntk(X)`
- Verify it is symmetric positive semidefinite
- Interpret it as a Gram matrix of parameter gradients

**Written part**:
- Explain why the linear-model NTK is exactly the Gram matrix of the input data.

In [24]:
# === Exercise 12: Scaffold ===

def linear_ntk(X):
    pass  # YOUR CODE HERE


# Build a small dataset and justify your PSD check rather than only printing it.

In [25]:
# === Exercise 12: Solution ===

def linear_ntk(X):
    X = np.asarray(X, dtype=float)
    return X @ X.T


X = np.array([
    [-1.0, 0.0],
    [-0.2, 1.0],
    [0.4, 1.2],
    [1.1, -0.3]
])
K = linear_ntk(X)
eigvals = np.linalg.eigvalsh(K)

print("K =\n", K)
print("symmetric?", np.allclose(K, K.T))
print("eigenvalues =", eigvals)
print("positive semidefinite?", np.all(eigvals >= -1e-10))

K =
 [[ 1.    0.2  -0.4  -1.1 ]
 [ 0.2   1.04  1.12 -0.52]
 [-0.4   1.12  1.6   0.08]
 [-1.1  -0.52  0.08  1.3 ]]
symmetric? True
eigenvalues = [-0.      0.      2.3919  2.5481]
positive semidefinite? True


---
## Solution Checks

This final cell runs lightweight checks on the solution implementations. It is intentionally narrower than the exercise prompts: the written parts and proof-style questions still need to be answered by the learner.

In [26]:
print("Running solution checks...\n")

# Exercise 1
assert np.isclose(dot_product([1, 2, -1], [3, 0, 1]), 2.0)
assert np.isclose(p_norm([3, -4], 2), 5.0)

# Exercise 2
assert np.isclose(cosine_similarity([1, 0], [0, 1]), 0.0)
assert np.isclose(angle_between([1, 0], [0, 1]), 90.0)

# Exercise 3
assert are_linearly_independent([np.array([1, 0]), np.array([0, 1])])
in_span, coeffs = is_in_span([np.array([1, 0]), np.array([0, 1])], np.array([5, -2]))
assert in_span and np.allclose(coeffs, [5, -2])

# Exercise 4
proj = project_onto(np.array([1.0, 2.0]), np.array([1.0, 0.0]))
assert np.allclose(proj, [1.0, 0.0])
Q = gram_schmidt([np.array([1.0, 0.0]), np.array([1.0, 1.0])])
assert np.allclose(Q @ Q.T, np.eye(2))

# Exercise 5
A_test = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
P_test = projection_matrix(A_test)
assert np.allclose(P_test @ P_test, P_test)
x_test = least_squares_solution(A_test, np.array([2.0, 1.0, 4.0]))
assert np.allclose(A_test.T @ (A_test @ x_test - np.array([2.0, 1.0, 4.0])), 0.0)

# Exercise 6
A6 = np.array([[1.0, 2.0, 3.0], [2.0, 4.0, 6.0], [1.0, 1.0, 2.0]])
rank, nullity = rank_nullity(A6)
assert rank + nullity == A6.shape[1]
N = null_space_basis(A6)
if N.size:
    assert np.allclose(A6 @ N, 0.0)

# Exercise 7
p = softmax([1.0, 2.0, 3.0])
assert is_in_simplex(p)

# Exercise 8
X8 = random_unit_vectors(50, 32)
mean_cos, std_cos = pairwise_cosine_stats(X8)
assert abs(mean_cos) < 0.15
assert std_cos > 0.0

# Exercise 9
X9 = np.random.randn(20, 50)
Y9 = jl_project(X9, 10, seed=0)
assert Y9.shape == (20, 10)

# Exercise 10
WQ = np.random.randn(16, 3)
WK = np.random.randn(16, 3)
assert attention_interaction_rank(WQ, WK) <= 3
B = np.random.randn(16, 2)
A = np.random.randn(2, 16)
assert np.linalg.matrix_rank(lora_update(B, A)) <= 2

# Exercise 11
assert np.allclose(soft_threshold(np.array([0.1, -0.4]), 0.2), np.array([0.0, -0.2]))
assert np.allclose(l2_shrink(np.array([1.0, -2.0]), 1.0), np.array([0.5, -1.0]))

# Exercise 12
X12 = np.array([[1.0, 0.0], [0.0, 1.0]])
K12 = linear_ntk(X12)
assert np.allclose(K12, np.eye(2))
assert np.all(np.linalg.eigvalsh(K12) >= -1e-10)

print("All solution checks passed.")

Running solution checks...

All solution checks passed.
